In [ ]:
cd ../..

## Feature Ideas

- User level features (row per user, timestamp):
    - `userId`
    - `timestamp`
    - `months_since_account_creation`
    - `display_name_is_null` (careful this isn't a leak!)
    - `website_url_is_null`
    - `about_me_length`
    - `location_is_null`
    - `num_engage`
    - `weeks_since_last_badge`
    - `badge_score`: sum of log(1 / badge_incidence) -- captures a total of engage weighted by their rarity. Eg: let's say a user has two engage, one that 5% of users have and another that 1% of users have, then the badge_score is log(20) + log(100)

- Post level features (row per user, timestamp, postid for posts created before timestamp) - for later aggregation:
    - `userId`
    - `timestamp`
    - `postId`
    - `post_type`
    - `post_rank`: to sort posts by age
    - `has_accepted_ans`: questions only
    - `is_accepted_ans`: answers only
    - `num_tags`: questions only
    - `title_length`: questions only
    - `body_length`
    - `days_since_last_post`
    - `num_postive_votes`
    - `num_negative_votes`
    - `num_comments`
    - `avg_comment_length`
    - `num_distinct_commenters`
    - `avg_commenter_badge_score`


- **Proposed feature set** (row per user, timestamp):
    - `* from user_features`
    - Question feats:
        - last_question (feats as of the last question asked by user, NULL if not exists):
            - `weeks_ago`: how many weeks ago did the user post their last question 
            - `has_accepted_ans`
            - `num_tags`
            - `title_length`
            - `body_length`
            - `num_postive_votes`
            - `num_negative_votes`
            - `num_comments`
            - `avg_comment_length`
            - `num_distinct_commenters`
            - `avg_commenter_badge_score`
        - questions_last_6mo (aggregate feats over questions asked by user in the last 6 months, NULL otherwise):
            - `num_questions_last_6mo`
            - `avg_has_accepted_ans`
            - `avg_num_tags`
            - `avg_title_length`
            - `avg_body_length`
            - `avg_num_postive_votes`
            - `avg_num_negative_votes`
            - `avg_num_comments`
            - `avg_avg_comment_length`
            - `avg_num_distinct_commenters`
            - `avg_commenter_badge_score`
    - Answer feats:
        - last_answer (feats as of the last answer posted by user, NULL if not exists)
            - `weeks_ago`: how many weeks ago did the user post their last answer 
            - `is_accepted_ans`
            - `body_length`
            - `num_postive_votes`
            - `num_negative_votes`
            - `num_comments`
            - `avg_comment_length`
            - `num_distinct_commenters`
            - `avg_commenter_badge_score`
        - answers_last_3mo (aggregate feats over answeres posted by user in the last 3 months, NULL otherwise):
            - `num_answeres_last_3mo`
            - `acceptance_rate`
            - `avg_body_length`
            - `avg_num_postive_votes`
            - `avg_num_negative_votes`
            - `avg_num_comments`
            - `avg_avg_comment_length`
            - `avg_num_distinct_commenters`
            - `avg_commenter_badge_score`

In [ ]:
import duckdb
from torch_frame.utils import infer_df_stype

import utils

conn = duckdb.connect('stack/stack.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

In [ ]:
with open('stack/user-engagement/feats.sql', 'r') as f:
    # run once with train_labels and once with val_labels
    template = f.read()

# create train, val and test features
# takes 1 - 5 mins
for s in ['train', 'val', 'test']:
    print(f'Creating {s} table')
    query = utils.render_jinja_sql(template, dict(set=s, subsample=0))
    conn.sql(query)
    print(f'{s} table created')

In [ ]:
utils.validate_feature_tables('user_engagement', conn)

In [ ]:
%%sql train_df <<
from user_engagement_train_feats

In [ ]:
infer_df_stype(train_df)

In [ ]:
utils.feature_summary_df(train_df.sample(20_000), 'contribution', classification=True)

## Feature Importances

In [ ]:
import numpy as np
import shap
from torch_frame import TaskType, stype
from torch_frame.data import Dataset
from torch_frame.gbdt import LightGBM

from inferred_stypes import task_to_stypes
from train_gbdt import TASK_PARAMS

TASK = 'rel-stack-user-engagement'

task_params = TASK_PARAMS[TASK]

In [ ]:
%%sql val_df <<
select * from user_engagement_val_feats;

In [ ]:
col_to_stype = task_to_stypes[TASK].copy()
val_tf = Dataset(
    val_df,
    col_to_stype=col_to_stype,
    target_col=task_params['target_col'],
).materialize().tensor_frame

In [ ]:
gbdt = LightGBM(task_type=task_params['task_type'])
gbdt.load(f'models/{TASK}_lgbm.json')
pred = gbdt.predict(tf_test=val_tf).numpy()

In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

sample = np.random.randint(0, len(val_tf), size=10_000)

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[sample])
shap_values = explainer.shap_values(val_arr, pred[sample])

# TODO verify
feat_names = val_tf.col_names_dict.get(stype.categorical, []) + val_tf.col_names_dict[stype.numerical]

shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=30, feature_names=feat_names)

In [ ]:
shap.dependence_plot('last_q_weeks_ago', shap_values, val_arr, feature_names=feat_names)

In [ ]:
conn.close()